### Topic modeling

Topic modeling sobre las queries y las sinopsis del historial--> Elegir coincidencias y similitudes
LDA o BERTopic?

In [ ]:
# Importar bibliotecas necesarias
import pandas as pd
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import word_tokenize
import re
import time
!pip install unidecode
from unidecode import unidecode
import string
from datasets import load_dataset
import nltk
nltk.download('punkt_tab')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 6.0 MB/s eta 0:00:0000:01


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [39]:
!pip install gensim

## Carga de datasets

Traemos el dataset de sinopsis de peliculas de IMDb desde Hugging Face

In [3]:
sinopsis = load_dataset("mathigatti/spanish_imdb_synopsis")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Traemos el dataset proporcionado con la información de los usuarios y sus respectivas queries

In [4]:
usuarios = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/usuarios/usuarios.csv")

In [5]:
usuarios

,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
0,U01,Valentina,definido,Durmiendo con su enemigo,Más allá de la muerte,Desaparecida,¡Olvídate de mí!,The Crazies,Quiero una película donde una mujer enfrenta u...
1,U02,Rodrigo,definido,Adiós Bafana,Mi pie izquierdo,L.A. Confidential,Érase una vez en América,El juego del halcón,Busco algo basado en hechos reales sobre corru...
2,U03,Camila,definido,Los padres de él,Mamá a la fuerza,Norbit,Elizabethtown,My Sassy Girl,Una comedia donde la relación entre dos person...
3,U04,Tomás,definido,Matrix,Fahrenheit 451,¡Olvídate de mí!,X-Men,El único,Algo que haga pensar sobre qué es real y qué e...
4,U05,Lucía,definido,La novia cadáver,Spirit: El corcel indomable,Las aventuras de Peabody y Sherman,Los Increíbles,Steamboy,Animación donde el protagonista lucha por su l...
5,U06,Martín,definido,Bienvenidos a Collinwood,El gran golpe,L.A. Confidential,Sympathy for Mr. Vengeance,La otra cara del crimen,Un grupo de personas planea un robo o estafa y...
6,U07,Sofía,definido,Velvet Goldmine,"Cuanto más, ¡mejor!",La vida de bohemia,Cero en conducta,Corazón salvaje,Una película sobre músicos o artistas que vive...
7,U08,Diego,definido,Superdetective en Hollywood,Mission: Impossible,Misión: Imposible 3,"Walker, Texas Ranger",300,Acción directa con un héroe que trabaja solo o...
8,U09,Elena,definido,Viaje a Darjeeling,Mi Idaho privado,Melinda y Melinda,La ciencia del sueño,Un beso,Algo tranquilo sobre personas que intentan rec...
9,U10,Facundo,definido,Sátántangó,Corazón salvaje,Mi Idaho privado,La ciencia del sueño,Sympathy for Mr. Vengeance,"Algo que sea difícil de clasificar, con una ló..."


Visualizamos las queries

In [6]:
for texto in usuarios['query']:
    print(texto)

Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Busco algo basado en hechos reales sobre corrupción o poder político
Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas
Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime
Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada
Una película sobre músicos o artistas que viven al margen, con mucha atmósfera y estilo visual
Acción directa con un héroe que trabaja solo o casi solo contra una organización criminal o corrupta
Algo tranquilo sobre personas que intentan reconectar o entenderse después de una distancia larga
Algo que sea difícil de clasificar, con una lógica narrativa propia, no convencional
No sé bien, algo que valga la pena ver un domingo a la noche, que enganche desde el princi

Construimos el dataset de peliculas, agregando un id que faltaba.

In [7]:
df_pelis = pd.DataFrame(sinopsis['train'])
df_pelis["id"] = df_pelis.index + 1
df_pelis.head()

,description,keywords,genre,year,name,director,id
0,"Orin Boyd, un duro policía de una comisaría de...","vietnam war veteran, heroína, drogas, narcotra...","acción, crimen, suspense",2001.0,Herida abierta,Andrzej Bartkowiak,1
1,Al llegar a un pequeño pueblo donde ha heredad...,"herencia, hostess, comedia negra, pueblo, magia","comedia, terror",1989.0,"Elvira, reina de las tinieblas",James Signorelli,2
2,Una mujer finge su muerte en un intento de esc...,"violencia doméstica, muerte fingida, borderlin...","drama, suspense",1991.0,Durmiendo con su enemigo,Joseph Ruben,3
3,Durante un memorial en la ciudad natal de su p...,"manic pixie dream girl, publicidad, bad public...","comedia, drama, romance",2005.0,Elizabethtown,Cameron Crowe,4
4,Las pruebas nucleares francesas irradian a una...,"monstruo gigante, iguana, militar, giant footp...","acción, ciencia ficción, suspense",1998.0,Godzilla,Roland Emmerich,5


# Preprocesado

In [10]:
def normalizar_texto(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto)
    texto = unidecode(texto)
    texto = re.sub(r"\d+", "", texto)
    texto = texto.lower()
    texto = re.sub(r"[^a-z]+", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

df_pelis['description'] = df_pelis['description'].apply(normalizar_texto)

In [11]:
# Normalizar y tokenizar manteniendo estructura original
df_pelis['description_tokenized'] = df_pelis['description'].apply(lambda x: x.split() if pd.notna(x) else [])

In [12]:
# Leer stopwords
stopwords = pd.read_csv('https://raw.githubusercontent.com/gefero/ecyt_lcd_intro_nlp/main/U3/data/stopwords.txt',
                       sep='\t',
                       names=['word'])
stopwords['word'] = stopwords['word'].apply(unidecode)

# # Agregar stopwords adicionales
# stopwords_adicionales = ['ano', 'anos', 'ohlala', 'foto', 'the']
# stopwords = pd.concat([stopwords, pd.DataFrame({'word': stopwords_adicionales})])

# Filtrar stopwords de los tokens
df_pelis['description_tokenized'] = df_pelis['description_tokenized'].apply(
   lambda x: [word for word in x if word not in stopwords['word'].values]
)

In [25]:
vectorizer = CountVectorizer() # se puede incluir todo el proceso de limpieza y tokenización dentro del vectorizador, también n-gramas, max_df y min_df para filtrar palabras por frecuencia.
matriz_doc_term = vectorizer.fit_transform(df_pelis['description_tokenized'].apply(lambda x: ' '.join(x)))

In [26]:
matriz_doc_term.shape

(4967, 14399)

In [27]:
nombres_features = vectorizer.get_feature_names_out()
nombres_features

array(['aames', 'aang', 'abadia', ..., 'zorg', 'zorro', 'zozobra'],
      dtype=object)

# Modelo

In [40]:
import gensim
from gensim.utils import simple_preprocess
from gensim import corpora

In [41]:
diccionario = corpora.Dictionary(df_pelis['description_tokenized'].tolist())

In [43]:
print(f"Vocabulario total: {len(diccionario)} palabras")

Vocabulario total: 14399 palabras


In [ ]:
#diccionario.filter_extremes(
#    no_below=2,    # eliminar tokens que aparecen en menos de 2 documentos
#    no_above=0.9,  # eliminar tokens que aparecen en más del 90% de los documentos
#    #keep_n=10000   # conservar solo las 10.000 palabras más frecuentes (opcional)
#)

# Podemos persistir el dictionary si queremos reutilizarlo
#diccionario.save('mi_diccionario.dict')
#diccionario = corpora.Dictionary.load('mi_diccionario.dict')

In [44]:
corpus_bow = [diccionario.doc2bow(doc) for doc in df_pelis['description_tokenized'].tolist()]

In [45]:
from gensim.models import LdaModel

In [46]:
modelo_lda = LdaModel(
    corpus=corpus_bow,
    id2word=diccionario,
    num_topics=10,
    passes=20,
    iterations=100,
    alpha='auto',
    eta='auto',
    random_state=12
)

In [47]:
for idx, topic in modelo_lda.show_topics(num_topics=10, num_words=10, formatted=False):
    print(f"Topic {idx+1}:")
    for word, prob in topic:
        print(f"    {word}: {prob:.4f}")

Topic 1:
    policia: 0.0261
    asesinato: 0.0154
    asesino: 0.0144
    detective: 0.0109
    serie: 0.0104
    ve: 0.0091
    agente: 0.0080
    investiga: 0.0068
    grupo: 0.0057
    caso: 0.0051
Topic 2:
    joven: 0.0138
    familia: 0.0093
    hombre: 0.0090
    mujer: 0.0058
    vida: 0.0058
    convierte: 0.0054
    siglo: 0.0052
    abogado: 0.0046
    esposa: 0.0041
    jefe: 0.0041
Topic 3:
    hijo: 0.0080
    vida: 0.0067
    mujer: 0.0059
    historia: 0.0059
    familia: 0.0058
    guerra: 0.0055
    anos: 0.0054
    quot: 0.0052
    amor: 0.0051
    vidas: 0.0048
Topic 4:
    mundo: 0.0065
    joven: 0.0061
    mujer: 0.0054
    vida: 0.0048
    convierte: 0.0048
    equipo: 0.0044
    hombre: 0.0039
    conoce: 0.0037
    policia: 0.0035
    casa: 0.0035
Topic 5:
    quot: 0.0435
    casa: 0.0060
    vida: 0.0058
    historia: 0.0052
    vive: 0.0049
    mujer: 0.0044
    anos: 0.0044
    joven: 0.0042
    amigos: 0.0041
    grupo: 0.0039
Topic 6:
    ciudad: 0.0094

In [38]:
from gensim.models import CoherenceModel

In [ ]:
K = range(3, 31)  # de 3 a 25 tópicos
resultados = []
for k in K:
    modelo = LdaModel(
        corpus=corpus_bow,
        id2word=diccionario,
        num_topics=k,
        passes=20,
        iterations=100,
        alpha='auto',
        eta='auto',
        random_state=42
    )

    # Coherence C_V
    coherencia = CoherenceModel(
        model=modelo,
        texts=df_pelis['description_tokenized'].tolist(),
        dictionary=diccionario,
        coherence='c_v'
    )

    c_v = coherencia.get_coherence()

    # Topic Diversity (N = 10)
    top_words_por_topico = [
        [word for word, _ in modelo.show_topic(k_idx, topn=10)]
        for k_idx in range(k)
    ]
    todas_las_palabras = [w for topico in top_words_por_topico for w in topico]
    td = len(set(todas_las_palabras)) / len(todas_las_palabras)

    resultados.append({'k': k, 'coherencia': c_v, 'diversidad': td})

df_resultados = pd.DataFrame(resultados)